In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

import faiss



c:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
df = pd.read_csv(
    "../data/processed/bindingdb_clean.csv"
)

df = df.sample(
    5000,
    random_state=42
).reset_index(drop=True)

print(df.shape)

(5000, 4)


smiles -> graph

In [3]:
def smiles_to_graph(
    smiles,
    target=0
):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    x = []

    for atom in mol.GetAtoms():

        x.append([
            atom.GetAtomicNum()
        ])

    edges = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edges.append([i, j])
        edges.append([j, i])

    if len(edges) == 0:
        return None

    return Data(
        x=torch.tensor(
            x,
            dtype=torch.float
        ),

        edge_index=torch.tensor(
            edges,
            dtype=torch.long
        ).t()
    )

model class


In [4]:
class AffinityGNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(1, 64)

        self.conv2 = GCNConv(64, 128)

        self.fc = nn.Linear(
            128,
            1
        )

    def encode(
        self,
        x,
        edge_index,
        batch
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = global_mean_pool(
            x,
            batch
        )

        return x

    def forward(
        self,
        x,
        edge_index,
        batch
    ):

        x = self.encode(
            x,
            edge_index,
            batch
        )

        return self.fc(x)

Load checkPoint

In [5]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = AffinityGNN().to(device)

model.load_state_dict(
    torch.load(
        "../models/checkpoints/pharmagpt_v0_7.pt",
        map_location=device
    )
)

model.eval()

print("Model Loaded")

Model Loaded


Generate Embedding

In [6]:
embeddings = []
smiles_list = []

with torch.no_grad():

    for smiles in df["Ligand SMILES"]:

        graph = smiles_to_graph(
            smiles
        )

        if graph is None:
            continue

        graph = graph.to(device)

        batch = torch.zeros(
            graph.num_nodes,
            dtype=torch.long
        ).to(device)

        emb = model.encode(
            graph.x,
            graph.edge_index,
            batch
        )

        embeddings.append(
            emb.cpu().numpy()[0]
        )

        smiles_list.append(
            smiles
        )

print(
    "Embeddings:",
    len(embeddings)
)

print(
    "Dimension:",
    embeddings[0].shape
)

Embeddings: 4953
Dimension: (128,)


Build FAISS index

In [7]:
vectors = np.array(
    embeddings,
    dtype=np.float32
)

index = faiss.IndexFlatL2(
    vectors.shape[1]
)

index.add(vectors)

print(
    "Vectors:",
    index.ntotal
)

Vectors: 4953


similarity Search

In [8]:
query_id = 0

distances, indices = index.search(
    vectors[query_id].reshape(
        1,
        -1
    ),
    10
)

for rank, idx in enumerate(indices[0]):

    print(
        rank,
        distances[0][rank]
    )

    print(
        smiles_list[idx]
    )

    print()

0 0.0
Nc1nc([O-])c2ncn([C@@H]3O[C@H](COP(O)(=O)OP(O)(=O)C[P+](O)(O)[O-])[C@@H](O)[C@H]3O)c2n1

1 2.6392677e-06
CC(C)N1C(CNCC#C)=Cc2cc(sc2S1(=O)=O)S(N)(=O)=O |c:9|

2 3.2652947e-06
O[C@H]1C[C@@H](O[C@@H]1COP(O)(=O)OP(O)(=O)OP(O)(O)=O)n1ccc2ccc(cc12)[N+]([O-])=O

3 1.1799901e-05
Nc1ccc2n(ccc2c1)[C@H]1C[C@H](O)[C@@H](COP(O)(=O)OP(O)(=O)OP(O)(O)=O)O1

4 3.9810107e-05
CN(c1ccc(cc1)-c1sc(C(O)=O)c(OCC(O)=O)c1Br)S(=O)(=O)c1ccc(cc1)C(F)(F)F

5 4.470308e-05
Oc1ccc(cc1Br)C1(OC(=O)c2cccc3cccc1c23)c1ccc(O)c(Br)c1

6 4.5390483e-05
Oc1ccc(cc1Br)C1(OC(=O)c2ccc3ccccc3c12)c1ccc(O)c(Br)c1

7 4.5390483e-05
Oc1ccc(cc1Br)C1(OC(=O)c2ccc3ccccc3c12)c1ccc(O)c(Br)c1

8 4.5390483e-05
Oc1ccc(cc1Br)C1(OC(=O)c2ccc3ccccc3c12)c1ccc(O)c(Br)c1

9 4.9521193e-05
COc1cc(C=C(C#N)C#N)cc(Br)c1O

